# GUI prototype — solara

Same UI as `gui_ipywidgets.ipynb`, expressed in [solara](https://solara.dev/). Runs inline in the notebook AND can be served as a standalone web app:

```bash
solara run experiments/gui_solara.py    # if exported to .py
```

Solara uses a React-like reactive component model: `solara.reactive` values trigger re-renders of any component that reads them. No manual `observe` callbacks.

Exploratory — see `experiments/README.md` for context.

In [ ]:
import solara
import shlex, subprocess

## Reactive state

In [ ]:
algo = solara.reactive('asp_mgm')
subpixel = solara.reactive(9)
processes = solara.reactive(4)
threads_mp = solara.reactive(2)

left_image = solara.reactive('left.tif')
right_image = solara.reactive('right.tif')
left_cam = solara.reactive('left.xml')
right_cam = solara.reactive('right.xml')
out_prefix = solara.reactive('out_stereo/run')

dry_run = solara.reactive(True)
run_output = solara.reactive('')

## Command builder

In [ ]:
def build_cmd():
    return [
        'parallel_stereo',
        '--stereo-algorithm', algo.value,
        '--subpixel-mode', str(subpixel.value),
        '--processes', str(processes.value),
        '--threads-multiprocess', str(threads_mp.value),
        left_image.value, right_image.value,
        left_cam.value, right_cam.value,
        out_prefix.value,
    ]

def on_run():
    cmd = build_cmd()
    if dry_run.value:
        run_output.value = 'DRY RUN — would execute:\n' + shlex.join(cmd)
        return
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        parts = [f'returncode: {result.returncode}']
        if result.stdout:
            parts.append('--- stdout (first 2 KB) ---')
            parts.append(result.stdout[:2000])
        if result.stderr:
            parts.append('--- stderr (first 2 KB) ---')
            parts.append(result.stderr[:2000])
        run_output.value = '\n'.join(parts)
    except FileNotFoundError as e:
        run_output.value = f'parallel_stereo not on PATH: {e}'

## App

In [ ]:
@solara.component
def Page():
    cmd_str = shlex.join(build_cmd())
    with solara.Column():
        with solara.Card('Stereo parameters'):
            solara.Select(label='Algorithm', value=algo, values=['asp_bm', 'asp_mgm', 'asp_sgm'])
            solara.Select(label='Subpixel mode', value=subpixel, values=[1, 2, 9])
            solara.SliderInt(label='Processes', value=processes, min=1, max=16)
            solara.SliderInt(label='Threads / process', value=threads_mp, min=1, max=8)
        with solara.Card('Inputs and output'):
            solara.InputText(label='Left image', value=left_image)
            solara.InputText(label='Right image', value=right_image)
            solara.InputText(label='Left camera XML', value=left_cam)
            solara.InputText(label='Right camera XML', value=right_cam)
            solara.InputText(label='Output prefix', value=out_prefix)
        with solara.Card('Generated CLI'):
            solara.Preformatted(cmd_str)
        with solara.Card('Run'):
            solara.Checkbox(label='Dry run (do not execute)', value=dry_run)
            solara.Button('Run', on_click=on_run, color='primary')
            if run_output.value:
                solara.Preformatted(run_output.value)

Page()

## Notes

- Reactive values (`solara.reactive`) auto-re-render readers; no `observe` callbacks needed.
- The same `Page` component runs as a standalone web app via `solara run script.py`. Useful for a no-Jupyter Codespace experience: open the forwarded port and you're in a browser-only UI.
- File-path inputs are plain text. Solara's [`FileBrowser`](https://solara.dev/api/file_browser) component is the obvious next step.
- Output is buffered until `subprocess.run` returns; for long-running commands swap to `subprocess.Popen` + `solara.use_thread` to stream progress.
- Compare the same prototype in ipywidgets: `experiments/gui_ipywidgets.ipynb`.